In [1]:
import requests
from bs4 import BeautifulSoup
import json
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin

BASE_URL = "https://115.159.159.82"
print("环境准备完成")


环境准备完成


In [2]:
api_url = BASE_URL + "/api/devices"

r = requests.get(
    api_url,
    params={"page": 1, "page_size": 50},
    timeout=10
)

print("状态码:", r.status_code)
print("实际请求地址:", r.url)

状态码: 200
实际请求地址: https://115.159.159.82/api/devices?page=1&page_size=50


In [3]:
data = r.json()

print(type(data))
print(data.keys())


<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [4]:
print("当前页   :", data["page"])
print("每页条数 :", data["page_size"])
print("设备总数 :", data["total"])
print("本页设备 :", len(data["items"]))
print()
print(data["items"][0])


当前页   : 1
每页条数 : 50
设备总数 : 5000
本页设备 : 50

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [5]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("设备总数 :", total)
print("每页条数 :", page_size)
print("总页数   :", total_pages)


设备总数 : 5000
每页条数 : 50
总页数   : 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(api_url, params={"page": page, "page_size": page_size}, timeout=10)
    page_data = r.json()
    devices.extend(page_data["items"])

print("共采集到 :", len(devices), "台设备")
print("唯一 device_id :", len(set(x["device_id"] for x in devices)))


共采集到 : 5000 台设备
唯一 device_id : 5000


In [7]:
current_url = BASE_URL + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")

print("状态码:", r.status_code)
print("本页企业数:", len(cards))


状态码: 200
本页企业数: 20


In [8]:
card = cards[0]

details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)


{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [9]:
company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)

print("编号:", company_id)
print("名称:", name)
print("状态:", status)


编号: C000001
名称: 临港精工001有限公司
状态: 运营中


In [10]:
industry = details["所属行业"]
zone = details["所在区域"]

device_count = int(details["设备数量"].replace("台", "").strip())

print("行业:", industry)
print("区域:", zone)
print("设备数:", device_count)


行业: 汽车零部件
区域: 智造一区
设备数: 6


In [11]:
def parse_card(card):
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details.get("所属行业"),
        "zone": details.get("所在区域"),
        "device_count": int(details.get("设备数量", "0").replace("台", "").strip()),
        "status": card.select_one(".badge").get_text(strip=True),
    }

print(parse_card(cards[0]))


{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


In [12]:
next_link = soup.select_one("nav.pagination a:last-of-type")

print(next_link.get_text(strip=True))
print(next_link.get("href"))


下一页
/companies?page=2


In [13]:
companies = []
page_counts = []

for i in range(3):
    r = requests.get(current_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    page_cards = soup.select("article.company-card")
    page_counts.append(len(page_cards))

    for card in page_cards:
        companies.append(parse_card(card))

    next_link = soup.select_one("nav.pagination a:last-of-type")
    current_url = urljoin(current_url, next_link.get("href"))

print("每页企业数:", page_counts)
print("合计:", len(companies))


每页企业数: [20, 20, 20]
合计: 60


In [14]:
print("=== 最终验证结果 ===")
print("Company 总数      :", len(companies))
print("唯一 company_id   :", len(set(c["company_id"] for c in companies)))
print("Device 总数       :", len(devices))
print("唯一 device_id    :", len(set(x["device_id"] for x in devices)))

device_counts = Counter(x["company_id"] for x in devices)

mismatch = 0
for c in companies:
    if c["device_count"] != device_counts.get(c["company_id"], 0):
        mismatch += 1

print("不一致企业数      :", mismatch)


=== 最终验证结果 ===
Company 总数      : 60
唯一 company_id   : 60
Device 总数       : 5000
唯一 device_id    : 5000
不一致企业数      : 0


In [15]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("已保存:", [p.name for p in sorted(data_dir.iterdir())])


已保存: ['companies.json', 'devices.json']
